# Toy binary kit (framework test)

**Kit:** `toy_binary` · **Template:** `binary_tabular` · **Prediction type:** `current_state`

> **Educational/research use only.** Educational/research output. Not a clinically validated risk score, diagnosis, or treatment recommendation.

> **SYNTHETIC DATA by default** — artificial, not clinically realistic, not derived from any
> participant data. Real All of Us data is used only inside the All of Us Researcher Workbench.

Cell banners: **Framework — do not edit** (shared infrastructure calls) · **Baseline — working
example** (copy it into your own cells to experiment) · **Your work** (runnable starter code).

## 0. Setup

Run these cells first. In Colab or local Jupyter keep `DATA_SOURCE = "synthetic"`. Inside the
**All of Us Researcher Workbench only**, a verified kit may use `DATA_SOURCE = "all_of_us"`;
anywhere else the framework refuses All of Us mode before touching any data.

In [ ]:
# Framework — do not edit
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_ml_kits") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "clinical-ml-kits @ git+https://github.com/CadenWondell/UNF-AIRO-Club.git@v0.1.0#subdirectory=projects/Fall_2026"])

In [ ]:
# Framework — do not edit
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from clinical_ml_kits import load_kit

DATA_SOURCE = "synthetic"  # "all_of_us" only inside the All of Us Researcher Workbench
SEED = 2026
pd.set_option("display.max_columns", 50)

kit = load_kit("toy_binary", data_source=DATA_SOURCE)

## 1. Clinical problem

**Clinical question.** Toy question for framework testing: given non-label-defining synthetic information, does the artificial record meet the toy phenotype?

No clinical content: artificial test kit.

> **PROVISIONAL — synthetic-only, not an approved clinical definition**
>
> The values below exist only so the kit runs on synthetic data. They are not approved clinical rules.

| Item | Provisional value | Verification |
|---|---|---|
| `cohort.inclusion[0].age_at_index.years` | `18.0` | pending V-21 |
| `temporal_policy.same_day` | `exclude_index_day` | pending V-20 |
| `temporal_policy.exclude_index_encounter` | `True` | pending V-20 |
| `temporal_policy.washout` | `P0D` | pending V-20 |
| `temporal_policy.lookback` | `P1095D` | pending V-20 |
| `phenotype.case_rule.has_event.min_count` | `2` | pending V-21 |
| `phenotype.control_rule.all_of[0].measurement.value` | `5.0` | pending V-21 |
| `phenotype.control_sampling.controls_per_case` | `2.0` | pending V-20 |
| `phenotype.min_observation_history` | `P365D` | pending V-21 |

In [ ]:
# Framework — do not edit
print(kit.describe())

## 2. Dataset and cohort

#### 1. Official documentation
None. Toy kit for framework tests only.
#### 2. Clinical / dataset definition
A fictional phenotype defined by fictional `SYN-FX-` codes. **[PROVISIONAL]**
#### 3. Logical concepts and dataset realizations
`fx_*` fixture concepts with synthetic realizations only.
#### 4. Cohort, cases, and controls
Age ≥ 18 (provisional); cases: two `fx_dx` on distinct days; controls: `fx_marker` < 5 and no `fx_dx`, two per case (random ratio). **[PROVISIONAL]**
#### 5. Index and cutoff rules
Case index: date the case rule is first met. Control index: one outpatient visit. Index-day observations and the index encounter excluded. **[PROVISIONAL]**
#### 6. Features and exclusions
See [features](features.md).
#### 7. Kit definition (configuration)
`kit.yaml` **[method decision]**.
#### 8. Notebook and baseline model
Generated Template A notebook; logistic regression baseline.

The framework builds the cohort, labels, index times, features, and patient-level
train/validation/test partitions for you. The **test partition is hidden** until the final
evaluation in section 12.

In [ ]:
# Framework — do not edit
data = kit.load()
data.summary()

In [ ]:
# Baseline — working example
print("Training rows:", len(data.train.X), "| features:", data.train.X.shape[1])
data.train.X.head()

**Features used as model input**

| Feature | Meaning | Derivations |
|---|---|---|
| `fx_age` | Age in years at the index time (fixture) | latest |
| `fx_bmi` | Body mass index (fixture) | latest, delta, mean |
| `fx_creatinine` | Creatinine (fixture) | latest, count |
| `fx_sodium` | Sodium (fixture) | latest |
| `fx_comorbidity` | Fixture comorbidity history | count |
| `fx_outpatient_visits` | Outpatient visits (fixture) | count (window P365D) |

**Deliberately excluded (label-defining or prohibited)**

- `fx_dx` — label-defining concept
- `fx_marker` — label-defining concept
- `fx_dx` — label-defining (toy case rule)
- `fx_marker` — label-defining (toy control evidence)

## 3. Exploratory data analysis

Explore the training partition only. Look at missingness carefully: a missing lab is *not* a
normal lab, and how often a test was ordered can itself carry information.

In [ ]:
# Your work
missing = data.train.X.isna().mean().sort_values(ascending=False)
missing.to_frame("fraction_missing").head(20)

In [ ]:
# Your work
print("Outcome rate in training data (sampled development prevalence):",
      round(data.train.y.mean(), 3))
data.train.X.describe().T

In [ ]:
# Your work
eda_columns = ["fx_bmi__latest", "fx_creatinine__latest", "fx_sodium__latest"]
fig, axes = plt.subplots(1, len(eda_columns), figsize=(4 * len(eda_columns), 3))
for ax, col in zip(np.atleast_1d(axes), eda_columns):
    for label, part in data.train.X.groupby(data.train.y.to_numpy()):
        ax.hist(part[col].dropna(), bins=30, alpha=0.5, density=True, label=f"label={label}")
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Your work
# Is missingness related to the outcome? (missing labs are informative, never "normal")
pd.DataFrame({
    "missing_rate_label_0": data.train.X[data.train.y.to_numpy() == 0].isna().mean(),
    "missing_rate_label_1": data.train.X[data.train.y.to_numpy() == 1].isna().mean(),
}).head(20)

## 4. Preprocessing

`make_preprocessor` builds the shared Template A preprocessing: median imputation **plus a
missingness indicator** for every numeric feature, then standard scaling; categorical features
get an explicit `"__missing__"` category and one-hot encoding. Inside a `Pipeline` it is fitted
on the training rows only.

In [ ]:
# Baseline — working example
from clinical_ml_kits.preprocessing import make_preprocessor

preprocessor = make_preprocessor(kit.feature_spec)
preprocessor

## 5. Baseline model

The baseline is a plain scikit-learn `Pipeline` with logistic regression. Its parameters come
from `kit.yaml`. Copy this cell into your own cells to experiment, so the baseline stays
available as a comparison point.

In [ ]:
# Baseline — working example
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline = Pipeline([
    ("preprocess", make_preprocessor(kit.feature_spec)),
    ("model", LogisticRegression(C=1.0, penalty="l2", class_weight=None, max_iter=1000, solver="lbfgs")),
])
baseline.fit(data.train.X, data.train.y)
val_proba = baseline.predict_proba(data.validation.X)[:, 1]
val_proba[:5]

## 6. Evaluate the baseline

Expected output: a metrics table (AUROC, AUPRC with the prevalence as its baseline,
sensitivity, specificity at the `youden` threshold chosen on validation), the
confusion matrix, and ROC / precision-recall curves.

In [ ]:
# Baseline — working example
from clinical_ml_kits.evaluation import evaluate_binary, plot_roc_pr

baseline_results = evaluate_binary(
    data.validation.y, val_proba, threshold_rule="youden", metrics=["auroc", "auprc", "sensitivity", "specificity", "precision", "recall", "f1", "brier"]
)
print("Confusion matrix:", baseline_results.confusion)
baseline_results.table()

In [ ]:
# Baseline — working example
plot_roc_pr(data.validation.y, val_proba, title="Baseline (validation)")

## 7. Your model

Try a different model. Use any scikit-learn estimator inside a `Pipeline` with the same
preprocessor. You never need to implement a learning algorithm yourself.

In [ ]:
# Your work
from sklearn.ensemble import RandomForestClassifier

forest = Pipeline([
    ("preprocess", make_preprocessor(kit.feature_spec)),
    ("model", RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEED)),
])
forest.fit(data.train.X, data.train.y)
forest_results = evaluate_binary(
    data.validation.y, forest.predict_proba(data.validation.X)[:, 1],
    threshold_rule="youden", metrics=["auroc", "auprc", "sensitivity", "specificity", "precision", "recall", "f1", "brier"],
)
forest_results.table()

In [ ]:
# Your work
# Other standard estimators to try (each runs as-is; edit and compare):
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

alternatives = {
    "gradient_boosting": HistGradientBoostingClassifier(max_iter=200, random_state=SEED),
    "svm": SVC(probability=True, C=1.0, random_state=SEED),
    "knn": KNeighborsClassifier(n_neighbors=25),
}
fitted_alternatives = {}
for name, estimator in alternatives.items():
    pipe = Pipeline([("preprocess", make_preprocessor(kit.feature_spec)), ("model", estimator)])
    fitted_alternatives[name] = pipe.fit(data.train.X, data.train.y)
sorted(fitted_alternatives)

## 8. Hyperparameter tuning

Hyperparameter tuning **must** use patient-grouped folds: `cv=kit.cv_splitter()` and
`groups=data.train.groups`. Default `cv=5` is not allowed in this project.

In [ ]:
# Your work
from sklearn.model_selection import GridSearchCV

cv = kit.cv_splitter()  # patient-grouped folds within the training partition
param_grid = {"model__n_estimators": [100, 300], "model__min_samples_leaf": [2, 10]}
search = GridSearchCV(forest, param_grid, cv=cv, scoring="average_precision")
search.fit(data.train.X, data.train.y, groups=data.train.groups)
best = search.best_estimator_
print("Best parameters:", search.best_params_, "| CV AUPRC:", round(search.best_score_, 3))

## 9. Model comparison

Compare models on the **validation** partition. Never use the test partition to choose
between models.

In [ ]:
# Your work
from clinical_ml_kits.evaluation import compare_models

candidates = {"baseline": baseline, "forest": forest, "tuned_forest": best,
              **fitted_alternatives}
comparison = compare_models(candidates, data.validation, metrics=["auroc", "auprc", "sensitivity", "specificity", "precision", "recall", "f1", "brier"],
                            threshold_rule="youden")
comparison.sort_values("auprc", ascending=False)

In [ ]:
# Your work
plot_roc_pr(
    data.validation.y,
    {name: model.predict_proba(data.validation.X)[:, 1] for name, model in candidates.items()},
    title="Validation comparison",
)

## 10. Calibration

**Sampled development prevalence vs population prevalence.** This kit samples controls
(strategy `random_ratio`, 2.0 controls per case), so the outcome rate in the data is the *sampled development prevalence*, not the
prevalence in any real population. Predicted probabilities are calibrated (at best) to that
development sample and must not be read as a patient's population disease probability or a
clinical risk score. Recalibration uses only the grouped out-of-fold helper
`kit.calibrate_grouped` (verification item V-14).

In [ ]:
# Baseline — working example
from clinical_ml_kits.evaluation import calibration_summary, plot_calibration

print(calibration_summary(data.validation.y, val_proba).describe())
plot_calibration(data.validation.y, val_proba, bins=10, strategy="quantile")

In [ ]:
# Your work
# Optional recalibration: grouped out-of-fold calibration fitted on the training partition only.
calibrated = kit.calibrate_grouped(best, method="sigmoid")
cal_proba = calibrated.predict_proba(data.validation.X)[:, 1]
print(calibration_summary(data.validation.y, cal_proba).describe())

## 11. Explainability

Explain the models: standardized logistic-regression coefficients (including missingness
indicators) and permutation importance on the validation partition.

In [ ]:
# Baseline — working example
from clinical_ml_kits.evaluation import coefficient_table

coefficient_table(baseline).head(15)

In [ ]:
# Your work
from clinical_ml_kits.evaluation import permutation_importance_table

permutation_importance_table(best, data.validation.X, data.validation.y, seed=SEED).head(15)

## 12. Final selection and completion

Choose your final model, then run the **single** logged evaluation on the held-out test
partition. Complete the model through the standard prediction interface.

#### Sampled development prevalence vs population prevalence
Two controls per case: probabilities reflect the sampled development prevalence, not a population.
#### Index-visit bias (residual)
Cases are indexed at diagnosis visits.
#### Provisional and unverified items
All values are provisional.

In [ ]:
# Your work
final_model = best  # choose and justify your final model using validation results only

In [ ]:
# Framework — do not edit
report = kit.final_evaluation(final_model)  # held-out test partition: once, logged
print("95% bootstrap intervals on test:", report.test_ci)
report.table()

In [ ]:
# Framework — do not edit
envelopes = kit.to_envelopes(final_model, data.validation.X.head(3))
print(envelopes[0].model_dump_json(indent=2))

In [ ]:
# Framework — do not edit
import tempfile
from pathlib import Path

record = kit.complete(final_model)
print(record.model_id, "| environment:", record.environment, "| portable:", record.portable)
if kit.environment == "synthetic":  # demo models are allowed only for synthetic data
    kit.save_model(final_model, Path(tempfile.gettempdir()) / "toy_binary_demo_model.joblib", portable=True)